In [1]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report

In [2]:
# ============================================================
# 0) CHARGEMENT DES DONNÉES
# ============================================================

df_all = pd.read_csv("../datasets/schaefcomb_Wang2023SimpleGSR_all.tsv", sep="\t")

print(df_all.diagnosis.value_counts())
print(df_all.site_id.value_counts())

# ============================================================
# PLOTS ÂGE CONTROL vs PATIENT
# ============================================================

control_age = df_all[df_all["diagnosis"] == "CONTROL"]["age"]
patient_age = df_all[df_all["diagnosis"] != "CONTROL"]["age"]

/tmp/ipykernel_818995/832223823.py:5: DtypeWarning: Columns (93967) have mixed types. Specify dtype option on import or set low_memory=False.
  df_all = pd.read_csv("../datasets/schaefcomb_Wang2023SimpleGSR_all.tsv", sep="\t")


diagnosis
CONTROL    1209
Autism      863
SCHZ        168
BIPOLAR      49
ADHD         40
Name: count, dtype: int64
site_id
ds000030          261
ABIDEII-KKI_1     211
NYU               184
ds_cobre          146
USM               101
ABIDEII-OHSU_1     91
ABIDEII-NYU_1      78
ABIDEII-GU_1       74
UCLA_1             73
ds004302           71
MAX_MUN            57
PITT               57
KKI                55
ABIDEII-IP_1       55
ABIDEII-SDSU_1     55
YALE               55
ABIDEII-BNI_1      54
ABIDEII-EMC_1      54
TRINITY            49
ABIDEII-ONRC_2     48
ABIDEII-TCD_1      42
ABIDEII-IU_1       40
CALTECH            38
ABIDEII-ETHZ_1     37
OLIN               36
SDSU               36
LEUVEN_2           35
ABIDEII-USM_1      32
ABIDEII-UCD_1      32
ABIDEII-UCLA_1     32
SBL                30
LEUVEN_1           29
OHSU               28
ABIDEII-KUL_3      27
UCLA_2             26
Name: count, dtype: int64


In [ ]:
# ============================================================
# 0bis) OPTIONS GLOBALES
# ============================================================

# Méthode de correction des confounds sur les connectomes
# "none"        : pas de correction de confounds
# "regression"  : régression (age, sexe)
# "combat"      : ComBat (site comme batch, age & sexe comme covariables)
CONF_METHOD = "combat"   # "none" / "regression" / "combat"

# PCA en entrée du classif
USE_PCA = True
N_PCS   = 9

# Cross-validation
N_SPLITS = 5  # k-fold

# Inclure ou non age/sex comme features dans le modèle final
INCLUDE_AGE_SEX_IN_MODEL = True

# ============================================================
# 1) FILTRER TRAIN / VAL ET PRÉPARER CIBLE
# ============================================================

# TRAIN : sujets de ds000030 + ds_cobre, uniquement CONTROL vs SCHZ
mask_train = (
    df_all["site_id"].isin(["ds000030", "ds_cobre"])
    & df_all["diagnosis"].isin(["CONTROL", "SCHZ"])
)

# VALIDATION inter-site : ds004302, uniquement CONTROL vs SCHZ
mask_val = (
    (df_all["site_id"] == "ds004302")
    & df_all["diagnosis"].isin(["CONTROL", "SCHZ"])
)

df_train = df_all.loc[mask_train].copy()
df_val   = df_all.loc[mask_val].copy()

print("Train (ds000030 + ds_cobre):", df_train.shape)
print("Validation (ds004302):", df_val.shape)
print("Train site_id counts:\n", df_train["site_id"].value_counts())
print("Val   site_id counts:\n", df_val["site_id"].value_counts())

# Cible binaire : 1 = SCHZ, 0 = CONTROL
y_train = (df_train["diagnosis"] == "SCHZ").astype(int)
y_val   = (df_val["diagnosis"] == "SCHZ").astype(int)

# Encodage sexe (0/1)
for d in (df_train, df_val):
    d["gender_num"] = (d["gender"] == "M").astype(float)

# S'assurer que age est float
df_train["age"] = df_train["age"].astype(float)
df_val["age"]   = df_val["age"].astype(float)

# ============================================================
# 2) CONNECTOMES : SÉLECTION + IMPUTATION SANS FUITE
# ============================================================

# Colonnes de connectome : celles qui commencent par "corr_"
corr_cols_all = [c for c in df_all.columns if c.startswith("corr_")]

# On retire les colonnes 100% NaN sur le TRAIN uniquement
nan_all_train = df_train[corr_cols_all].isna().all()
corr_cols = nan_all_train[~nan_all_train].index.tolist()
print(f"Colonnes corr_ retenues : {len(corr_cols)}")

# Matrices brutes connectomes
X_corr_train_raw = df_train[corr_cols].copy()
X_corr_val_raw   = df_val[corr_cols].copy()

# Imputation des NaN par la moyenne (fit sur TRAIN uniquement → pas de fuite)
corr_imputer = SimpleImputer(strategy="mean")
X_corr_train_imp = corr_imputer.fit_transform(X_corr_train_raw)
X_corr_val_imp   = corr_imputer.transform(X_corr_val_raw)

# On retransforme en DataFrame pour garder index / noms
X_corr_train_imp = pd.DataFrame(X_corr_train_imp, index=df_train.index, columns=corr_cols)
X_corr_val_imp   = pd.DataFrame(X_corr_val_imp,   index=df_val.index,   columns=corr_cols)

# ============================================================
# 3) CORRECTION POUR LES VARIABLES CONFONDANTES
# ============================================================

def residualize_confounds(X_train, X_test, conf_train, conf_test):
    """
    Régression linéaire multi-sortie :
        X = conf * B + erreur
    On ajuste B sur le TRAIN, puis on enlève conf*B pour TRAIN & TEST.
    """
    # On ajoute un intercept
    C_train = np.column_stack([np.ones(len(conf_train)), conf_train.values])
    C_test  = np.column_stack([np.ones(len(conf_test)),  conf_test.values])

    # B = (C^T C)^-1 C^T X  (multi-feature)
    B = np.linalg.pinv(C_train).dot(X_train.values)

    # prédictions
    X_train_hat = C_train.dot(B)
    X_test_hat  = C_test.dot(B)

    # résidus = données "déconfondées"
    X_train_resid = X_train.values - X_train_hat
    X_test_resid  = X_test.values - X_test_hat

    X_train_resid = pd.DataFrame(X_train_resid, index=X_train.index, columns=X_train.columns)
    X_test_resid  = pd.DataFrame(X_test_resid,  index=X_test.index,  columns=X_test.columns)

    return X_train_resid, X_test_resid


if CONF_METHOD == "regression":
    # On corrige les connectomes pour Age + Sexe
    conf_train = df_train[["age", "gender_num"]]
    conf_val   = df_val[["age", "gender_num"]]
    X_corr_train_corr, X_corr_val_corr = residualize_confounds(
        X_corr_train_imp, X_corr_val_imp,
        conf_train, conf_val
    )
    print("Correction par régression (age, sexe) appliquée.")

elif CONF_METHOD == "combat":
    # ⚠️ Nécessite : pip install neuroHarmonize
    from neuroHarmonize import harmonizationLearn, harmonizationApply

    # On concatène train + val pour l’harmonisation ComBat
    X_corr_all_imp = np.vstack([X_corr_train_imp.values, X_corr_val_imp.values])

    covars_all = pd.DataFrame({
        "SITE":  np.concatenate([df_train["site_id"].values, df_val["site_id"].values]),
        "AGE":   np.concatenate([df_train["age"].values,      df_val["age"].values]),
        "SEX_M": np.concatenate([df_train["gender_num"].values,
                                 df_val["gender_num"].values]),
    })

    print("Apprentissage ComBat sur (train + val) pour corriger l'effet SITE...")
    combat_model, X_corr_all_adj = harmonizationLearn(
        X_corr_all_imp,
        covars_all,
        smooth_terms=["AGE"]
    )

    # On sépare à nouveau train / val
    n_train = len(df_train)
    X_corr_train_corr = pd.DataFrame(
        X_corr_all_adj[:n_train, :],
        index=df_train.index,
        columns=corr_cols
    )
    X_corr_val_corr = pd.DataFrame(
        X_corr_all_adj[n_train:, :],
        index=df_val.index,
        columns=corr_cols
    )
    print("Correction ComBat (site) appliquée.")

else:
    # Pas de correction de confounds sur les connectomes
    X_corr_train_corr = X_corr_train_imp
    X_corr_val_corr   = X_corr_val_imp
    print("Aucune correction de confounds appliquée sur les connectomes.")

# ============================================================
# 4) CONSTRUCTION DU DESIGN FINAL POUR LE CLASSIFIEUR
# ============================================================

if INCLUDE_AGE_SEX_IN_MODEL:
    X_train_final = pd.concat(
        [X_corr_train_corr, df_train[["age", "gender_num"]]],
        axis=1
    )
    X_val_final = pd.concat(
        [X_corr_val_corr, df_val[["age", "gender_num"]]],
        axis=1
    )
else:
    X_train_final = X_corr_train_corr.copy()
    X_val_final   = X_corr_val_corr.copy()

print("Shape X_train_final :", X_train_final.shape)
print("Shape X_val_final   :", X_val_final.shape)
print("len(y_train)        :", len(y_train))
print("len(y_val)          :", len(y_val))

# ============================================================
# 5) PIPELINE PCA + LOGISTIC REGRESSION
# ============================================================

if USE_PCA:
    feature_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=N_PCS, random_state=42))
    ])
else:
    feature_pipeline = Pipeline([
        ("scaler", StandardScaler())
    ])

clf = Pipeline([
    ("features", feature_pipeline),
    ("logreg", LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        C=1.0,
        class_weight="balanced",
        max_iter=5000,
        n_jobs=-1
    ))
])

# ============================================================
# 6) CROSS-VALIDATION SUR TRAIN (ds000030 + ds_cobre)
# ============================================================

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    clf,
    X_train_final,
    y_train,
    cv=skf,
    scoring="roc_auc",
    n_jobs=-1
)

print(f"\n=== CV {N_SPLITS}-fold (ds000030 + ds_cobre) ===")
print(f"AUC ROC CV : {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")

# ============================================================
# 7) ENTRAÎNEMENT FINAL + VALIDATION INTER-SITE
# ============================================================

clf.fit(X_train_final, y_train)

y_proba_val = clf.predict_proba(X_val_final)[:, 1]
y_pred_val  = clf.predict(X_val_final)

print("\n=== VALIDATION INTER-SITE (ds004302) ===")
print("AUC ROC :", roc_auc_score(y_val, y_proba_val))
print(classification_report(y_val, y_pred_val, target_names=["CONTROL", "SCHZ"]))